# Testing models from https://iopscience.iop.org/article/10.1088/1361-6560/ad9133#pmbad9133app2

In [1]:
import torch
import torch.nn as nn
import subprocess
import torch.nn.functional as F

### usefull links on model coding
about dataloaders https://pytorch.org/tutorials/beginner/data_loading_tutorial.html 

check points and early stopping https://machinelearningmastery.com/managing-a-pytorch-training-process-with-checkpoints-and-early-stopping/

CVAE MNIST https://github.com/debtanu177/CVAE_MNIST/blob/master/train_cvae.py

conditional VAE https://github.com/unnir/cVAE/blob/master/cvae.py

3D CNN https://github.com/xmuyzz/3D-CNN-PyTorch/blob/master/models/C3DNet.py
& https://machinelearningmastery.com/building-a-convolutional-neural-network-in-pytorch/

In [2]:
class ConvBlock(nn.Module):
    def __init__(self, nb_filters_in, nb_filters_out):
        """
        Initializes the convolutional block with convolution, batch normalization, and Leaky ReLU activation.

        Parameters:
        - nb_filters (int): Number of filters for the convolutional layer.
        """
        super(ConvBlock, self).__init__()
        self.padding = (3 // 2, 3 // 2)
        self.conv = nn.Conv2d(in_channels=nb_filters_in, out_channels=nb_filters_out, kernel_size=(3, 3), padding=self.padding, stride=2)
        #self.normalize = nn.BatchNorm2d(nb_filters_out)
        self.activation = nn.LeakyReLU()
        #self.activation = nn.ReLU()

    def forward(self, x):
        xhat = self.conv(x)
        #xhat = self.normalize(xhat)
        xhat = self.activation(xhat)
        return xhat


class DeconvBlock(nn.Module):
    def __init__(self,  nb_filters_in, nb_filters_out):
        """
        Initializes the deconvolutional block with transposed convolution, batch normalization, and Leaky ReLU activation.

        Parameters:
        - nb_filters (int): Number of filters for the transposed convolutional layer.
        """
        super(DeconvBlock, self).__init__()
        self.padding = (3 // 2, 3 // 2)
        self.deconv = nn.ConvTranspose2d(in_channels=nb_filters_in, out_channels=nb_filters_out, kernel_size=(3, 3), padding=self.padding, stride=2)
        #self.normalize = nn.BatchNorm2d(nb_filters_out)
        self.activation = nn.LeakyReLU()
        #self.activation = nn.ReLU()

    def forward(self, x):
        xhat = self.deconv(x)
        #xhat = self.normalize(xhat)
        xhat = self.activation(xhat)
        return xhat


'''def compute_kl(z_mean, z_log_var):
    """
    Computes the Kullback-Leibler divergence loss.

    Parameters:
    - z_mean (Tensor): Mean of the latent distribution.
    - z_log_var (Tensor): Log variance of the latent distribution.

    Returns:
    - kl_loss (Tensor): Computed KL divergence loss.
    """
    kl_loss = -0.5 * (1 + z_log_var - z_mean**2 - torch.exp(z_log_var))
    kl_loss = kl_loss.mean(dim=0).sum()
    return kl_loss
'''

'def compute_kl(z_mean, z_log_var):\n    """\n    Computes the Kullback-Leibler divergence loss.\n\n    Parameters:\n    - z_mean (Tensor): Mean of the latent distribution.\n    - z_log_var (Tensor): Log variance of the latent distribution.\n\n    Returns:\n    - kl_loss (Tensor): Computed KL divergence loss.\n    """\n    kl_loss = -0.5 * (1 + z_log_var - z_mean**2 - torch.exp(z_log_var))\n    kl_loss = kl_loss.mean(dim=0).sum()\n    return kl_loss\n'

In [3]:
class ImagePadding_4dtensor:
    def __init__(self, image):
        """
        Initializes the padding class with a 4D tensor (batch_size, channels, height, width).
        """
        # Ensure that the input is a 4D tensor (batch_size, channels, height, width)
        assert len(image.shape) == 4, "Input image must have shape (batch_size, channels, height, width)"
        self.image = image

    def pad_to_size(self, target_size):
        """
        Pads the input 2D image batch to the target size by adding pixels with zero values.

        Parameters:
        - target_size (tuple): The desired output size (height, width)

        Returns:
        - Padded 4D tensor (batch_size, channels, target_height, target_width)
        """
        current_height, current_width = self.image.shape[2], self.image.shape[3]

        # Calculate padding needed for each dimension
        pad_height = max(target_size[0] - current_height, 0)
        pad_width = max(target_size[1] - current_width, 0)

        # Divide padding into two sides (before and after)
        pad_top = pad_height // 2
        pad_bottom = pad_height - pad_top
        pad_left = pad_width // 2
        pad_right = pad_width - pad_left

        # Apply padding using torch.nn.functional.pad
        padded_image = F.pad(
            self.image,
            (pad_left, pad_right, pad_top, pad_bottom),  # (left, right, top, bottom)
            mode='constant',
            value=0  # Pad with zeros
        )

        return padded_image


In [4]:
class ImagePadding_3dtensor:
    def __init__(self, image):
        """
        Initializes the padding class with a 3D tensor (batch_size, height, width).
        """
        # Ensure that the input is a 3D tensor (batch_size, height, width)
        assert len(image.shape) == 3, "Input image must have shape (batch_size, height, width)"
        self.image = image

    def pad_to_size(self, target_size):
        """
        Pads the input 2D image batch to the target size by adding pixels with zero values.

        Parameters:
        - target_size (tuple): The desired output size (height, width)

        Returns:
        - Padded 3D tensor (batch_size, target_height, target_width)
        """
        current_height, current_width = self.image.shape[1], self.image.shape[2]

        # Calculate padding needed for each dimension
        pad_height = max(target_size[0] - current_height, 0)
        pad_width = max(target_size[1] - current_width, 0)

        # Divide padding into two sides (before and after)
        pad_top = pad_height // 2
        pad_bottom = pad_height - pad_top
        pad_left = pad_width // 2
        pad_right = pad_width - pad_left

        # Apply padding using torch.nn.functional.pad
        padded_image = F.pad(
            self.image.unsqueeze(1),  # Add channel dimension (B, 1, H, W) for padding
            (pad_left, pad_right, pad_top, pad_bottom),  # (left, right, top, bottom)
            mode='constant',
            value=0  # Pad with zeros
        ).squeeze(1)  # Remove extra channel dimension to get back (B, H, W)

        return padded_image




In [5]:
class ImageCropping_4dtensor:
    def __init__(self, image):
        """
        Initializes the cropping class with a 4D tensor (batch_size, channels, height, width).
        Assumes input shape is (batch_size, channels, height, width).
        """
        # Ensure that the input has a batch dimension and has 2 channels
        assert len(image.shape) == 4, "Input image must have shape (batch_size, channels, height, width)"
        self.image = image

    def crop_to_size(self, target_size):
        """
        Crops the input 2D image batch to the target size.

        Parameters:
        - target_size (tuple): The desired output size (height, width)

        Returns:
        - Cropped 4D tensor (batch_size, channels, target_height, target_width)
        """
        current_height, current_width = self.image.shape[2], self.image.shape[3]

        # Ensure the target size is smaller or equal to the current size
        assert target_size[0] <= current_height and target_size[1] <= current_width, \
            "Target size must be smaller or equal to the current size in both dimensions."

        # Calculate cropping indices
        crop_top = (current_height - target_size[0]) // 2
        crop_left = (current_width - target_size[1]) // 2

        # Crop the image while keeping the batch and channel structure
        cropped_image = self.image[:, :, crop_top:crop_top + target_size[0], crop_left:crop_left + target_size[1]]

        return cropped_image


In [6]:
class ImageCropping_3dtensor:
    def __init__(self, image):
        """
        Initializes the cropping class with a 3D tensor (batch_size, height, width).
        Assumes input shape is (batch_size, height, width).
        """
        # Ensure the input has a batch dimension
        assert len(image.shape) == 3, "Input image must have shape (batch_size, height, width)"
        self.image = image

    def crop_to_size(self, target_size):
        """
        Crops the input 2D image batch to the target size.

        Parameters:
        - target_size (tuple): The desired output size (height, width)

        Returns:
        - Cropped 3D tensor (batch_size, target_height, target_width)
        """
        current_height, current_width = self.image.shape[1], self.image.shape[2]

        # Ensure the target size is smaller or equal to the current size
        assert target_size[0] <= current_height and target_size[1] <= current_width, \
            "Target size must be smaller or equal to the current size in both dimensions."

        # Calculate cropping indices
        crop_top = (current_height - target_size[0]) // 2
        crop_left = (current_width - target_size[1]) // 2

        # Crop the image while keeping the batch structure
        cropped_image = self.image[:, crop_top:crop_top + target_size[0], crop_left:crop_left + target_size[1]]

        return cropped_image




In [7]:
# Model 
# code was adapted from https://github.com/ValentinGautierCreatis/occipet/blob/standard_dlr/src/deep_learning/vae1_2.py
class VAE_1modality(nn.Module):
    
    def __init__(self, feature_size, latent_size, in_channels=1):
        
        super(VAE_1modality_PET, self).__init__()
        
        self.feature_size = feature_size
        self.latent_size = latent_size
        self.in_channels = in_channels

        # encode
        #self.pool= 0
        
        self.conv = nn.Sequential(
            ConvBlock(1,32), 
            ConvBlock(32,64)
        )
        
        self.flatten = nn.Flatten()
        self.dense_mean = nn.Linear(self.feature_size, self.latent_size)  
        self.dense_log_var = nn.Linear(self.feature_size, self.latent_size)  
        #self.sampling = Sampling()
        
        self.dense = nn.Linear(self.latent_size, self.feature_size)  # Adjust size accordingly
        self.deconv = nn.Sequential(
            DeconvBlock(64,32),  # Assuming DeconvBlock is defined elsewhere
            DeconvBlock(32,32),
            DeconvBlock(32,32),
            nn.ConvTranspose2d(in_channels=32, out_channels=1, kernel_size=3, stride=1, padding=1),
            nn.ReLU()
        )




    def encode(self, x): # with condition - def encode(self, x, c):
        '''
        x: input scan (bs, 1, 197, 233, 189)
        c: condition (bs, class_size)
        '''
        #print(x.shape)
        image_padder = ImagePadding_4dtensor(x)
        
        # Target size [256, 256, 256]
        #target_size = (243, 243, 243)
        target_size = (256, 256)

        # Get the padded image
        padded_image = image_padder.pad_to_size(target_size)
        #print(padded_image.shape)
        #print('encoding')
        x = self.conv(padded_image)
        
        #print(x.shape)
        self.remember_shape = x.shape
        x = self.flatten(x)
        #print(x.shape)
        z_mu = self.dense_mean(x)
        z_var = self.dense_log_var(x)

       
        
        return z_mu, z_var

    def reparameterize(self, mu, logvar):
        #print('reparameterize')
        std = torch.exp(0.5*logvar)
        eps = torch.randn_like(std)
        return mu + eps*std

    def decode(self, z): # def decode(self, z, c):
        '''
        z: (bs, latent_size)
        c: (bs, class_size)
        '''
        #print('decoder')
        #inputs = torch.cat([z, c], 1) # (bs, latent_size+class_size)
        
        x = self.dense(z)
        #print(x.shape)
        x = torch.reshape(x,(self.remember_shape[0],self.remember_shape[1],self.remember_shape[2],self.remember_shape[3]))
        
        #print(x.shape)
        
        x = self.deconv(x)
        #x = torch.sigmoid(x) #!!
        
        #x = self.deconv_1ch(x)
        #print(x.shape)
        image_cropper = ImageCropping_4dtensor(x)
        
        # Target size [182, 218, 182]
        target_size = (182, 218)

        # Get the cropped image
        cropped_image = image_cropper.crop_to_size(target_size)
        #print(cropped_image.shape)
        return cropped_image #torch.cat([cropped_image_pet, cropped_image_mri], dim=1)

    def forward(self, x): #def forward(self, x, c):
        
        mu, logvar = self.encode(x) #self.encode(x, c)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar #self.decode(z, c), mu, logvar


In [8]:
# Model 
# code was adapted from https://github.com/ValentinGautierCreatis/occipet/blob/standard_dlr/src/deep_learning/vae1_2.py
class VAE_1modality_PET(nn.Module):
    
    def __init__(self, feature_size, latent_size, in_channels=1):
        
        super(VAE_1modality_PET, self).__init__()
        
        self.feature_size = feature_size
        self.latent_size = latent_size
        self.in_channels = in_channels

        # encode
        #self.pool= 0
        
        self.conv = nn.Sequential(
            ConvBlock(1,32), 
            ConvBlock(32,64)
        )
        
        self.flatten = nn.Flatten()
        self.dense_mean = nn.Linear(self.feature_size, self.latent_size)  
        self.dense_log_var = nn.Linear(self.feature_size, self.latent_size)  
        #self.sampling = Sampling()
        
        self.dense = nn.Linear(self.latent_size, self.feature_size)  # Adjust size accordingly
        self.deconv = nn.Sequential(
            DeconvBlock(64,32),  # Assuming DeconvBlock is defined elsewhere
            DeconvBlock(32,32),
            DeconvBlock(32,32),
            nn.ConvTranspose2d(in_channels=32, out_channels=1, kernel_size=3, stride=1, padding=1),
            nn.ReLU()
        )




    def encode(self, x): # with condition - def encode(self, x, c):
        '''
        x: input scan (bs, 1, 197, 233, 189)
        c: condition (bs, class_size)
        '''
        #print(x.shape)
        image_padder = ImagePadding_4dtensor(x)
        
        # Target size [256, 256, 256]
        #target_size = (243, 243, 243)
        target_size = (256, 256)

        # Get the padded image
        padded_image = image_padder.pad_to_size(target_size)
        #print(padded_image.shape)
        #print('encoding')
        x = self.conv(padded_image)
        
        #print(x.shape)
        self.remember_shape = x.shape
        x = self.flatten(x)
        #print(x.shape)
        z_mu = self.dense_mean(x)
        z_var = self.dense_log_var(x)

       
        
        return z_mu, z_var

    def reparameterize(self, mu, logvar):
        #print('reparameterize')
        std = torch.exp(0.5*logvar)
        eps = torch.randn_like(std)
        return mu + eps*std

    def decode(self, z): # def decode(self, z, c):
        '''
        z: (bs, latent_size)
        c: (bs, class_size)
        '''
        #print('decoder')
        #inputs = torch.cat([z, c], 1) # (bs, latent_size+class_size)
        
        x = self.dense(z)
        #print(x.shape)
        x = torch.reshape(x,(self.remember_shape[0],self.remember_shape[1],self.remember_shape[2],self.remember_shape[3]))
        
        #print(x.shape)
        
        x = self.deconv(x)
        #x = torch.sigmoid(x) #!!
        
        #x = self.deconv_1ch(x)
        #print(x.shape)
        image_cropper = ImageCropping_4dtensor(x)
        
        # Target size [182, 218, 182]
        target_size = (182, 218)

        # Get the cropped image
        cropped_image = image_cropper.crop_to_size(target_size)
        #print(cropped_image.shape)
        return cropped_image #torch.cat([cropped_image_pet, cropped_image_mri], dim=1)

    def forward(self, x): #def forward(self, x, c):
        
        mu, logvar = self.encode(x) #self.encode(x, c)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar #self.decode(z, c), mu, logvar


In [9]:
'''device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = VAE_1modality_PET(64*64*64, latent_size =1).to(device)
sum([param.nelement() for param in model.parameters()])'''

1104643

In [10]:
'''from torchsummary import summary
help(summary)
summary(model, [(1,182, 218)] , device='cpu') #[(2,182, 218, 182),(2,)]
'''

Help on function summary in module torchsummary.torchsummary:

summary(model, input_size, batch_size=-1, device='cuda')

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 32, 128, 128]             320
         LeakyReLU-2         [-1, 32, 128, 128]               0
         ConvBlock-3         [-1, 32, 128, 128]               0
            Conv2d-4           [-1, 64, 64, 64]          18,496
         LeakyReLU-5           [-1, 64, 64, 64]               0
         ConvBlock-6           [-1, 64, 64, 64]               0
           Flatten-7               [-1, 262144]               0
            Linear-8                    [-1, 1]         262,145
            Linear-9                    [-1, 1]         262,145
           Linear-10               [-1, 262144]         524,288
  ConvTranspose2d-11         [-1, 32, 127, 127]          18,464
        LeakyReLU-12         [-1, 32, 127, 127

In [11]:
# Model 
# code was adapted from https://github.com/ValentinGautierCreatis/occipet/blob/standard_dlr/src/deep_learning/vae1_2.py
class VAE_1modality_MRI(nn.Module):
    
    def __init__(self, feature_size, latent_size, in_channels=1):
        
        super(VAE_1modality_MRI, self).__init__()
        
        self.feature_size = feature_size
        self.latent_size = latent_size
        self.in_channels = in_channels

        # encode
        #self.pool= 0
        
        self.conv = nn.Sequential(
            ConvBlock(1,32), 
            ConvBlock(32,64), 
            ConvBlock(64,128),
            ConvBlock(128,256)
        )
        
        self.flatten = nn.Flatten()
        self.dense_mean = nn.Linear(self.feature_size, self.latent_size)  
        self.dense_log_var = nn.Linear(self.feature_size, self.latent_size)  
        #self.sampling = Sampling()
        
        self.dense = nn.Linear(self.latent_size, self.feature_size)  # Adjust size accordingly
        self.deconv = nn.Sequential(
            DeconvBlock(256,128),  # Assuming DeconvBlock is defined elsewhere
            DeconvBlock(128,128),
            DeconvBlock(128,64),
            DeconvBlock(64,64),
            DeconvBlock(64,32),
            nn.ConvTranspose2d(in_channels=32, out_channels=1, kernel_size=3, stride=1, padding=1),
            nn.ReLU()
        )




    def encode(self, x): # with condition - def encode(self, x, c):
        '''
        x: input scan (bs, 1, 197, 233, 189)
        c: condition (bs, class_size)
        '''
        #print(x.shape)
        image_padder = ImagePadding_4dtensor(x)
        
        # Target size [256, 256, 256]
        #target_size = (243, 243, 243)
        target_size = (256, 256)

        # Get the padded image
        padded_image = image_padder.pad_to_size(target_size)
        #print(padded_image.shape)
        #print('encoding')
        x = self.conv(padded_image)
        
        #print(x.shape)
        self.remember_shape = x.shape
        x = self.flatten(x)
        #print(x.shape)
        z_mu = self.dense_mean(x)
        z_var = self.dense_log_var(x)

       
        
        return z_mu, z_var

    def reparameterize(self, mu, logvar):
        #print('reparameterize')
        std = torch.exp(0.5*logvar)
        eps = torch.randn_like(std)
        return mu + eps*std

    def decode(self, z): # def decode(self, z, c):
        '''
        z: (bs, latent_size)
        c: (bs, class_size)
        '''
        #print('decoder')
        #inputs = torch.cat([z, c], 1) # (bs, latent_size+class_size)
        
        x = self.dense(z)
        #print(x.shape)
        x = torch.reshape(x,(self.remember_shape[0],self.remember_shape[1],self.remember_shape[2],self.remember_shape[3]))
        
        #print(x.shape)
        
        x = self.deconv(x)
        #x = torch.sigmoid(x) #!!
        
        #x = self.deconv_1ch(x)
        #print(x.shape)
        image_cropper = ImageCropping_4dtensor(x)
        
        # Target size [182, 218, 182]
        target_size = (182, 218)

        # Get the cropped image
        cropped_image = image_cropper.crop_to_size(target_size)
        #print(cropped_image.shape)
        return cropped_image #torch.cat([cropped_image_pet, cropped_image_mri], dim=1)

    def forward(self, x): #def forward(self, x, c):
        
        mu, logvar = self.encode(x) #self.encode(x, c)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar #self.decode(z, c), mu, logvar


In [12]:
'''device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = VAE_1modality_MRI(256*16*16, latent_size =1).to(device)
sum([param.nelement() for param in model.parameters()])'''

'device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")\nmodel = VAE_1modality_MRI(256*16*16, latent_size =1).to(device)\nsum([param.nelement() for param in model.parameters()])'

In [13]:
'''from torchsummary import summary
help(summary)
summary(model, [(1,182, 218)] , device='cpu') #[(2,182, 218, 182),(2,)]
'''

"from torchsummary import summary\nhelp(summary)\nsummary(model, [(1,182, 218)] , device='cpu') #[(2,182, 218, 182),(2,)]\n"

In [14]:
# Model 
# code was adapted from https://github.com/ValentinGautierCreatis/occipet/blob/standard_dlr/src/deep_learning/vae1_2.py
class VAE1_2(nn.Module):
    
    def __init__(self, feature_size, latent_size, in_channels=2):
        
        super(VAE1_2, self).__init__()
        
        self.feature_size = feature_size
        self.latent_size = latent_size
        self.in_channels = in_channels

        # encode
        #self.pool= 0
        
        self.conv = nn.Sequential(
            ConvBlock(2,32), 
            ConvBlock(32,64)
        )
        
        self.flatten = nn.Flatten()
        self.dense_mean = nn.Linear(self.feature_size, self.latent_size)  
        self.dense_log_var = nn.Linear(self.feature_size, self.latent_size)  
        #self.sampling = Sampling()
        
        self.dense = nn.Linear(self.latent_size, self.feature_size)  # Adjust size accordingly
        self.deconv = nn.Sequential(
            DeconvBlock(64,32),  # Assuming DeconvBlock is defined elsewhere
            DeconvBlock(32,32),
            nn.ConvTranspose2d(in_channels=32, out_channels=2, kernel_size=3, stride=1, padding=1)
        )

        self.deconv_1ch = nn.Sequential(
            DeconvBlock(64,32),  # Assuming DeconvBlock is defined elsewhere
            DeconvBlock(32,32),
            nn.ConvTranspose2d(in_channels=32, out_channels=1, kernel_size=3, stride=1, padding=1)
        )

        



    def encode(self, x): # with condition - def encode(self, x, c):
        '''
        x: input scan (bs, 1, 197, 233, 189)
        c: condition (bs, class_size)
        '''
        #print(x.shape)
        image_padder = ImagePadding_4dtensor(x)
        
        # Target size [256, 256, 256]
        #target_size = (243, 243, 243)
        target_size = (256, 256)

        # Get the padded image
        padded_image = image_padder.pad_to_size(target_size)
        #print(padded_image.shape)
        #print('encoding')
        x = self.conv(padded_image)
        
        #print(x.shape)
        self.remember_shape = x.shape
        x = self.flatten(x)
        #print(x.shape)
        z_mu = self.dense_mean(x)
        z_var = self.dense_log_var(x)

       
        
        return z_mu, z_var

    def reparameterize(self, mu, logvar):
        #print('reparameterize')
        std = torch.exp(0.5*logvar)
        eps = torch.randn_like(std)
        return mu + eps*std

    def decode(self, z): # def decode(self, z, c):
        '''
        z: (bs, latent_size)
        c: (bs, class_size)
        '''
        #print('decoder')
        #inputs = torch.cat([z, c], 1) # (bs, latent_size+class_size)
        
        x = self.dense(z)
        #print(x.shape)
        x = torch.reshape(x,(self.remember_shape[0],self.remember_shape[1],self.remember_shape[2],self.remember_shape[3]))
        
        #print(x.shape)
        
        x = self.deconv(x)
        #x = self.deconv_1ch(x)
        #print(x.shape)
        image_cropper = ImageCropping_4dtensor(x)
        
        # Target size [182, 218, 182]
        target_size = (182, 218)

        # Get the cropped image
        cropped_image = image_cropper.crop_to_size(target_size)
        #print(cropped_image.shape)
        return cropped_image #torch.cat([cropped_image_pet, cropped_image_mri], dim=1)

    def forward(self, x): #def forward(self, x, c):
        
        mu, logvar = self.encode(x) #self.encode(x, c)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar #self.decode(z, c), mu, logvar


In [15]:
'''device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = VAE1_2(64*64*64, latent_size =10).to(device)
sum([param.nelement() for param in model.parameters()])'''

'device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")\nmodel = VAE1_2(64*64*64, latent_size =10).to(device)\nsum([param.nelement() for param in model.parameters()])'

In [16]:
'''from torchsummary import summary
help(summary)
summary(model, [(2,182, 218)] , device='cpu') #[(2,182, 218, 182),(2,)]
'''

"from torchsummary import summary\nhelp(summary)\nsummary(model, [(2,182, 218)] , device='cpu') #[(2,182, 218, 182),(2,)]\n"

In [17]:
class Encoder(nn.Module):
    """Early fusion encoder for VAE."""
    def __init__(self, feature_size, latent_size):
        super(Encoder, self).__init__()
        self.conv = nn.Sequential(
            ConvBlock(2,32), 
            ConvBlock(32,64)
        )
        
        self.flatten = nn.Flatten()
        self.dense_mean = nn.Linear(feature_size, latent_size)  
        self.dense_log_var = nn.Linear(feature_size, latent_size)  
        #self.sampling = Sampling()
        
    def forward(self, x):
        image_padder = ImagePadding_4dtensor(x)
        
        # Target size [256, 256, 256]
        #target_size = (243, 243, 243)
        target_size = (256, 256)

        # Get the padded image
        padded_image = image_padder.pad_to_size(target_size)
        #print(padded_image.shape)
        #print('encoding')
        x = self.conv(padded_image)
        
        #print(x.shape)
        remember_shape = x.shape
        x = self.flatten(x)
        #print(x.shape)
        z_mu = self.dense_mean(x)
        z_var = self.dense_log_var(x)
        return z_mu, z_var, remember_shape


class Decoder(nn.Module):
    """A single decoder for VAE."""
    def __init__(self,  feature_size, latent_size):
        super(Decoder, self).__init__()
        self.dense = nn.Linear(latent_size, feature_size)  # Adjust size accordingly

        self.deconv_1ch = nn.Sequential(
            DeconvBlock(64,32),  # Assuming DeconvBlock is defined elsewhere
            DeconvBlock(32,32),
            nn.ConvTranspose2d(in_channels=32, out_channels=1, kernel_size=3, stride=1, padding=1)
        )
        
    def forward(self, z,remember_shape):
        x = self.dense(z)
        #print(x.shape)
        x = torch.reshape(x,(remember_shape[0],remember_shape[1],remember_shape[2],remember_shape[3]))
        
        #print(x.shape)
        
        x = self.deconv_1ch(x)
        #x = self.deconv_1ch(x)
        #print(x.shape)
        image_cropper = ImageCropping_4dtensor(x)
        
        # Target size [182, 218, 182]
        target_size = (182, 218)

        # Get the cropped image
        cropped_image = image_cropper.crop_to_size(target_size)
        #print(cropped_image.shape)
        return cropped_image #torch.cat([cropped_image_pet, cropped_image_mri], dim=1)



In [18]:
# Model 
# code was adapted from https://github.com/ValentinGautierCreatis/occipet/blob/standard_dlr/src/deep_learning/vae1_2.py
class VAE_multimodel(nn.Module):
    
    def __init__(self, feature_size, latent_size, in_channels=2):
        
        super(VAE_multimodel, self).__init__()
        
        self.feature_size = feature_size
        self.latent_size = latent_size
        self.in_channels = in_channels

        self.encoder = Encoder(feature_size, latent_size)
        self.decoders = nn.ModuleList([Decoder(feature_size, latent_size) for _ in range(2)])


        

    
    def reparameterize(self, mu, logvar):
        #print('reparameterize')
        std = torch.exp(0.5*logvar)
        eps = torch.randn_like(std)
        return mu + eps*std

   
    def forward(self, x): #def forward(self, x, c):
        
        mu, logvar, remember_shape = self.encoder(x) #self.encode(x, c)
        z = self.reparameterize(mu, logvar)
        reconstruction = torch.cat([self.decoders[0](z,remember_shape), self.decoders[1](z,remember_shape)], dim=1)
        return reconstruction, mu, logvar #self.decode(z, c), mu, logvar


In [19]:
# Model 
# code was adapted from https://github.com/ValentinGautierCreatis/occipet/blob/standard_dlr/src/deep_learning/vae1_2.py
class VAE_late_fusion(nn.Module):
    
    def __init__(self, feature_size, latent_size, in_channels=2):
        
        super(VAE_late_fusion, self).__init__()
        
        self.feature_size = feature_size
        self.latent_size = latent_size
        self.in_channels = in_channels

        # encode
        #self.pool= 0
        
        self.conv = nn.Sequential(
            ConvBlock(1,32), 
            ConvBlock(32,64)
        )
        
        self.flatten = nn.Flatten()
        self.dense_mean = nn.Linear(self.feature_size, self.latent_size)  
        self.dense_log_var = nn.Linear(self.feature_size, self.latent_size)  
        #self.sampling = Sampling()
        
        self.dense = nn.Linear(2*self.latent_size, 2*self.feature_size)  # Adjust size accordingly
        self.deconv = nn.Sequential(
            DeconvBlock(128,64),  # Assuming DeconvBlock is defined elsewhere
            DeconvBlock(64,32),
            DeconvBlock(32,32),
            nn.ConvTranspose2d(in_channels=32, out_channels=1, kernel_size=3, stride=1, padding=1),
            nn.ReLU()
        )

        



    def encode(self, x): # with condition - def encode(self, x, c):
        '''
        x: input scan (bs, 1, 197, 233, 189)
        c: condition (bs, class_size)
        '''
        #print(x.shape)
        image_padder = ImagePadding_4dtensor(x)
        
        # Target size [256, 256, 256]
        #target_size = (243, 243, 243)
        target_size = (256, 256)

        # Get the padded image
        padded_image = image_padder.pad_to_size(target_size)
        #print(padded_image.shape)
        #print('encoding')
        x = self.conv(padded_image)
        
        #print(x.shape)
        self.remember_shape = x.shape
        x = self.flatten(x)
        #print(x.shape)
        z_mu = self.dense_mean(x)
        z_var = self.dense_log_var(x)
        z = self.reparameterize(z_mu, z_var)

       
        
        return z_mu, z_var, z

    def reparameterize(self, mu, logvar):
        #print('reparameterize')
        std = torch.exp(0.5*logvar)
        eps = torch.randn_like(std)
        return mu + eps*std

    def decode(self, z): # def decode(self, z, c):
        '''
        z: (bs, latent_size)
        c: (bs, class_size)
        '''
        #print('decoder')
        #inputs = torch.cat([z, c], 1) # (bs, latent_size+class_size)
        
        
        x = self.dense(z)
        #print(x.shape)
        x = torch.reshape(x,(self.remember_shape[0],2*self.remember_shape[1],self.remember_shape[2],self.remember_shape[3]))
        
        #print(x.shape)
        
        x = self.deconv(x)
        #print(x.shape)
        image_cropper = ImageCropping_4dtensor(x)
        
        # Target size [182, 218, 182]
        target_size = (182, 218)

        # Get the cropped image
        cropped_image = image_cropper.crop_to_size(target_size)
        #print(cropped_image.shape)
        return cropped_image

    def forward(self, x): #def forward(self, x, c):
        
        mu, logvar, z = self.encode(x) #self.encode(x, c)
        return self.decode(z), mu, logvar #self.decode(z, c), mu, logvar


In [34]:
# Model 
# code was adapted from https://github.com/ValentinGautierCreatis/occipet/blob/standard_dlr/src/deep_learning/vae1_2.py
class MVAE_1modality_PET(nn.Module):
    
    def __init__(self, feature_size, latent_size, in_channels=1):
        
        super(MVAE_1modality_PET, self).__init__()
        
        self.feature_size = feature_size
        self.latent_size = latent_size
        self.in_channels = in_channels

        # encode
        #self.pool= 0
        
        self.conv = nn.Sequential(
            ConvBlock(1,32), 
            ConvBlock(32,64)
        )
        
        self.flatten = nn.Flatten()
        self.dense_mean = nn.Linear(self.feature_size, self.latent_size)  
        self.dense_log_var = nn.Linear(self.feature_size, self.latent_size)  
        #self.sampling = Sampling()
        
        self.dense = nn.Linear(2*self.latent_size, 2*self.feature_size)  # Adjust size accordingly
        self.deconv = nn.Sequential(
            DeconvBlock(128,64),  # Assuming DeconvBlock is defined elsewhere
            DeconvBlock(64,32),
            DeconvBlock(32,32),
            nn.ConvTranspose2d(in_channels=32, out_channels=1, kernel_size=3, stride=1, padding=1),
            nn.ReLU()
        )




    def encode(self, x): # with condition - def encode(self, x, c):
        '''
        x: input scan (bs, 1, 197, 233, 189)
        c: condition (bs, class_size)
        '''
        #print(x.shape)
        image_padder = ImagePadding_4dtensor(x)
        
        # Target size [256, 256, 256]
        #target_size = (243, 243, 243)
        target_size = (256, 256)

        # Get the padded image
        padded_image = image_padder.pad_to_size(target_size)
        #print(padded_image.shape)
        #print('encoding')
        x = self.conv(padded_image)
        
        #print(x.shape)
        self.remember_shape = x.shape
        x = self.flatten(x)
        #print(x.shape)
        z_mu = self.dense_mean(x)
        z_var = self.dense_log_var(x)

       
        
        return z_mu, z_var

    def reparameterize(self, mu, logvar):
        #print('reparameterize')
        std = torch.exp(0.5*logvar)
        eps = torch.randn_like(std)
        return mu + eps*std

    def decode(self, z): # def decode(self, z, c):
        '''
        z: (bs, latent_size)
        c: (bs, class_size)
        '''
        #print('decoder')
        #inputs = torch.cat([z, c], 1) # (bs, latent_size+class_size)
        
        x = self.dense(z)
        #print(x.shape)
        x = torch.reshape(x,(self.remember_shape[0],2*self.remember_shape[1],self.remember_shape[2],self.remember_shape[3]))
        
        #print(x.shape)
        
        x = self.deconv(x)
        #x = torch.sigmoid(x) #!!
        
        #x = self.deconv_1ch(x)
        #print(x.shape)
        image_cropper = ImageCropping_4dtensor(x)
        
        # Target size [182, 218, 182]
        target_size = (182, 218)

        # Get the cropped image
        cropped_image = image_cropper.crop_to_size(target_size)
        #print(cropped_image.shape)
        return cropped_image #torch.cat([cropped_image_pet, cropped_image_mri], dim=1)

    def forward(self, x): #def forward(self, x, c):
        
        mu, logvar = self.encode(x) #self.encode(x, c)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar #self.decode(z, c), mu, logvar


In [35]:
#device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
#model = MVAE(64*64*64, latent_size =1).to(device)
#model = MVAE_1modality_PET(feature_size = 64*64*64, latent_size =1).to(device)
#sum([param.nelement() for param in model.parameters()])

2217763

In [33]:
#from torchsummary import summary
#help(summary)
#summary(model, [(2,182, 218)] , device='cpu') #[(2,182, 218, 182),(2,)]

Help on function summary in module torchsummary.torchsummary:

summary(model, input_size, batch_size=-1, device='cuda')



RuntimeError: Given groups=1, weight of size [32, 1, 3, 3], expected input[2, 2, 256, 256] to have 1 channels, but got 2 channels instead

In [21]:
# Model 
# code was adapted from https://github.com/ValentinGautierCreatis/occipet/blob/standard_dlr/src/deep_learning/vae1_2.py
class MVAE_1modality_MRI(nn.Module):
    
    def __init__(self, feature_size, latent_size, in_channels=1):
        
        super(MVAE_1modality_MRI, self).__init__()
        
        self.feature_size = feature_size
        self.latent_size = latent_size
        self.in_channels = in_channels

        # encode
        #self.pool= 0
        
        self.conv = nn.Sequential(
            ConvBlock(1,32), 
            ConvBlock(32,64), 
            ConvBlock(64,128),
            ConvBlock(128,256)
        )
        
        self.flatten = nn.Flatten()
        self.dense_mean = nn.Linear(self.feature_size, self.latent_size)  
        self.dense_log_var = nn.Linear(self.feature_size, self.latent_size)  
        #self.sampling = Sampling()
        
        self.dense = nn.Linear(2*self.latent_size, 2*self.feature_size)  # Adjust size accordingly
        self.deconv = nn.Sequential(
            DeconvBlock(512,256),  # Assuming DeconvBlock is defined elsewhere
            DeconvBlock(256,128),
            DeconvBlock(128,64),
            DeconvBlock(64,64),
            DeconvBlock(64,32),
            nn.ConvTranspose2d(in_channels=32, out_channels=1, kernel_size=3, stride=1, padding=1),
            nn.ReLU()
        )




    def encode(self, x): # with condition - def encode(self, x, c):
        '''
        x: input scan (bs, 1, 197, 233, 189)
        c: condition (bs, class_size)
        '''
        #print(x.shape)
        image_padder = ImagePadding_4dtensor(x)
        
        # Target size [256, 256, 256]
        #target_size = (243, 243, 243)
        target_size = (256, 256)

        # Get the padded image
        padded_image = image_padder.pad_to_size(target_size)
        #print(padded_image.shape)
        #print('encoding')
        x = self.conv(padded_image)
        
        #print(x.shape)
        self.remember_shape = x.shape
        x = self.flatten(x)
        #print(x.shape)
        z_mu = self.dense_mean(x)
        z_var = self.dense_log_var(x)

       
        
        return z_mu, z_var

    def reparameterize(self, mu, logvar):
        #print('reparameterize')
        std = torch.exp(0.5*logvar)
        eps = torch.randn_like(std)
        return mu + eps*std

    def decode(self, z): # def decode(self, z, c):
        '''
        z: (bs, latent_size)
        c: (bs, class_size)
        '''
        #print('decoder')
        #inputs = torch.cat([z, c], 1) # (bs, latent_size+class_size)
        
        x = self.dense(z)
        #print(x.shape)
        x = torch.reshape(x,(self.remember_shape[0],2*self.remember_shape[1],self.remember_shape[2],self.remember_shape[3]))
        
        #print(x.shape)
        
        x = self.deconv(x)
        #x = torch.sigmoid(x) #!!
        
        #x = self.deconv_1ch(x)
        #print(x.shape)
        image_cropper = ImageCropping_4dtensor(x)
        
        # Target size [182, 218, 182]
        target_size = (182, 218)

        # Get the cropped image
        cropped_image = image_cropper.crop_to_size(target_size)
        #print(cropped_image.shape)
        return cropped_image #torch.cat([cropped_image_pet, cropped_image_mri], dim=1)

    def forward(self, x): #def forward(self, x, c):
        
        mu, logvar = self.encode(x) #self.encode(x, c)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar #self.decode(z, c), mu, logvar


In [22]:
# Model 
# code was adapted from https://github.com/ValentinGautierCreatis/occipet/blob/standard_dlr/src/deep_learning/vae1_2.py
class MVAE(nn.Module):
    
    def __init__(self, feature_size_pet,feature_size_mri, latent_size, in_channels=1):
        
        super(MVAE, self).__init__()
        
        self.feature_size_pet = feature_size_pet
        self.feature_size_mri = feature_size_mri
        self.latent_size = latent_size
        self.in_channels = in_channels

        # Create one VAE per modality and store in ModuleList
        self.nb_modalities = 2
        '''self.vaes = nn.ModuleList([
            VAE_late_fusion(feature_size = feature_size, latent_size=latent_size, in_channels = in_channels)
            for _ in range(self.nb_modalities)
        ])'''
        self.vae_pet = MVAE_1modality_PET(feature_size = feature_size_pet, latent_size=latent_size, in_channels = in_channels)
        self.vae_mri = MVAE_1modality_MRI(feature_size = feature_size_mri, latent_size=latent_size, in_channels = in_channels)

        

        
    def reparameterize(self, mu, logvar):
        #print('reparameterize')
        std = torch.exp(0.5*logvar)
        eps = torch.randn_like(std)
        return mu + eps*std
    
    def poe(self, means, log_vars):
        """
        Product of Experts (PoE) for Variational Autoencoder (VAE) fusion.
    
        Args:
            means (list of torch.Tensor): List of mean tensors from different experts.
            log_vars (list of torch.Tensor): List of log variance tensors from different experts.
    
        Returns:
            torch.Tensor: Combined mean
            torch.Tensor: Combined log variance
        """
        Ts = [torch.exp(-torch.clamp(lv, min=-10, max=10)) for lv in log_vars]
        #Ts = [torch.exp(-lv) for lv in log_vars]  # Precision (1/variance)
        log_var = -torch.log(sum(Ts)+ 1e-8)  # Combined log variance
    
        mean_numerator = sum(Ti * mean_i for Ti, mean_i in zip(Ts, means))
        mean = mean_numerator * torch.exp(log_var)  # Combined mean
    
        return mean, log_var


    

    def forward(self, x): #def forward(self, x, c):
        #print(x.shape)
        #print(x[0].shape)
        pet, mri = torch.split(x, 1, dim=1)
        #print(pet.shape)
        mu_pet, logvar_pet = self.vae_pet.encode(pet) #self.encode(x, c)
        mu_mri, logvar_mri = self.vae_mri.encode(mri)
        
        z_pet = self.reparameterize(mu_pet, logvar_pet)
        z_mri = self.reparameterize(mu_mri, logvar_mri)
        
        # Do I need “prior expert” (torch.zeros_like)? 
        #mu, log_var = self.poe([mu_pet, mu_mri,torch.zeros_like(mu_pet)],[logvar_pet, logvar_mri,torch.zeros_like(logvar_pet)])
        mu, log_var = self.poe([mu_pet, mu_mri],[logvar_pet, logvar_mri])

        #z = self.reparameterize(mu, log_var)
        
        z = torch.cat([z_pet,z_mri], dim=1)
        #recon_zpet = torch.cat([self.vaes[0].decode(z_pet), self.vaes[1].decode(z_pet)], dim=1)
        #recon_zmri = torch.cat([self.vaes[0].decode(z_mri), self.vaes[1].decode(z_mri)], dim=1)
        #recon_zpoe = torch.cat([self.vaes[0].decode(z), self.vaes[1].decode(z)], dim=1)
        recon_pet = self.vae_pet.decode(z)
        recon_mri = self.vae_mri.decode(z)
        recon = torch.cat([recon_pet, recon_mri], dim=1)
        #print(recon_zpoe.shape)
        #return recon_zpet, mu_pet, logvar_pet,recon_zmri,mu_mri,logvar_mri,recon_zpoe,mu,log_var #self.decode(z, c), mu, logvar
        #print(recon_z.shape)
        return recon, recon_pet, recon_mri



In [23]:
'''device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
#model = MVAE(64*64*64, latent_size =1).to(device)
model = MVAE(feature_size_pet = 64*64*64,feature_size_mri = 256*16*16, latent_size =1).to(device)
sum([param.nelement() for param in model.parameters()])'''

'device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")\n#model = MVAE(64*64*64, latent_size =1).to(device)\nmodel = MVAE(feature_size_pet = 64*64*64,feature_size_mri = 256*16*16, latent_size =1).to(device)\nsum([param.nelement() for param in model.parameters()])'

In [24]:
'''from torchsummary import summary
help(summary)
summary(model, [(2,182, 218)] , device='cpu') #[(2,182, 218, 182),(2,)]'''

"from torchsummary import summary\nhelp(summary)\nsummary(model, [(2,182, 218)] , device='cpu') #[(2,182, 218, 182),(2,)]"

In [25]:
#!jupyter nbconvert --to script addl_models.ipynb

In [26]:
# MRI 2D VAE from https://github.com/voanna/slices-to-3d-brain-vae/blob/master/experiments/MICCAI-release-version/main_experiment_256.py


class ResBlockUp(nn.Module):
    def __init__(self, filters_in, filters_out, act=True):
        super(ResBlockUp, self).__init__()
        self.act = act
        self.conv1_block = nn.Sequential(
            nn.Conv2d(filters_in, filters_in, 3, stride=1, padding=1),
            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.BatchNorm2d(filters_in),
            nn.LeakyReLU(0.2, inplace=True))

        self.conv2_block = nn.Sequential(
            nn.Conv2d(filters_in, filters_out, 3, stride=1, padding=1),
            nn.BatchNorm2d(filters_out))

        self.conv3_block = nn.Sequential(
            nn.Conv2d(filters_in, filters_out, 3, stride=1, padding=1),
            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.BatchNorm2d(filters_out),
            nn.LeakyReLU(0.2, inplace=True))

        self.lrelu = nn.LeakyReLU(0.2, inplace=False)

    def forward(self, x):
        conv1 = self.conv1_block(x)
        conv2 = self.conv2_block(conv1)
        if self.act:
            conv2 = self.lrelu(conv2)
        conv3 = self.conv3_block(x)
        if self.act:
            conv3 = self.lrelu(conv3)

        return conv2 + conv3
    
class ResBlockDown(nn.Module):
    def __init__(self, filters_in, filters_out, act=True):
        super(ResBlockDown, self).__init__()
        self.act = act
        self.conv1_block = nn.Sequential(
            nn.Conv2d(filters_in, filters_in, 3, stride=2, padding=1),
            nn.BatchNorm2d(filters_in),
            nn.LeakyReLU(0.2, inplace=True))

        self.conv2_block = nn.Sequential(
            nn.Conv2d(filters_in, filters_out, 3, stride=1, padding=1),
            nn.BatchNorm2d(filters_out))

        self.conv3_block = nn.Sequential(
            nn.Conv2d(filters_in, filters_out, 3, stride=2, padding=1),
            nn.BatchNorm2d(filters_out)
        )
        self.lrelu = nn.LeakyReLU(0.2, inplace=False)

    def forward(self, x):
        conv1 = self.conv1_block(x)
        conv2 = self.conv2_block(conv1)
        if self.act:
            conv2 = self.lrelu(conv2)
        conv3 = self.conv3_block(x)
        if self.act:
            conv3 = self.lrelu(conv3)

        return conv2 + conv3

    
class Encoder(nn.Module):
    def __init__(self, n_channels, gf_dim=16):
        super(Encoder, self).__init__()

        gf_dim = gf_dim

        self.conv1_block = nn.Sequential(
            nn.Conv2d(n_channels, gf_dim, 3, stride=1, padding=1),
            nn.BatchNorm2d(gf_dim),
            nn.LeakyReLU(0.2, inplace=True))

        self.res1 = ResBlockDown(gf_dim * 1, gf_dim * 1)
        self.res2 = ResBlockDown(gf_dim * 1, gf_dim * 2)
        self.res3 = ResBlockDown(gf_dim * 2, gf_dim * 4)
        self.res4 = ResBlockDown(gf_dim * 4, gf_dim * 8)
        self.res5 = ResBlockDown(gf_dim * 8, gf_dim * 16)

        self.res6 = ResBlockDown(gf_dim * 16, gf_dim * 32)
        self.res7 = ResBlockDown(gf_dim * 32, gf_dim * 64, act=False)
        self.res2_stdev = ResBlockDown(gf_dim * 32, gf_dim * 64, act=False)

    def encode(self, x):
        conv1 = self.conv1_block(x)
        z = self.res1(conv1)
        z = self.res2(z)
        z = self.res3(z)
        z = self.res4(z)
        z = self.res5(z)
        z = self.res6(z)
        z_mean = self.res7(z)

        z_std = self.res2_stdev(z)
        
        return z_mean, z_std
    
    def reparameterize(self, mu, std):
        std = torch.exp(std)
        eps = torch.randn_like(std)
        return mu + eps*std

    def forward(self, x):
        mu, std = self.encode(x)
        z = self.reparameterize(mu, std)
        return z, mu, std
    
class Decoder(nn.Module):
    def __init__(self, n_channels, gf_dim=16):
        super(Decoder, self).__init__()


        self.res0 = ResBlockUp(gf_dim*64, gf_dim * 32)
        self.res1 = ResBlockUp(gf_dim*32, gf_dim * 16)
        self.res2 = ResBlockUp(gf_dim*16, gf_dim * 8)
        self.res3 = ResBlockUp(gf_dim*8, gf_dim * 4)
        self.res4 = ResBlockUp(gf_dim*4, gf_dim * 2)
        self.res5 = ResBlockUp(gf_dim*2, gf_dim * 1)
        self.res6 = ResBlockUp(gf_dim*1, gf_dim * 1)
        self.conv_1_block = nn.Sequential(
            nn.Conv2d(gf_dim, gf_dim, 3, stride=1, padding=1),
            nn.BatchNorm2d(gf_dim),
            nn.LeakyReLU(0.2, inplace=True))


        self.conv2 = nn.Conv2d(gf_dim, n_channels, 3, stride=1, padding=1)

    def forward(self, z):
            x = self.res0(z)
            x = self.res1(x)
            x = self.res2(x)
            x = self.res3(x)
            x = self.res4(x)
            x = self.res5(x)
            x = self.res6(x)
            x = self.conv_1_block(x)
            x = self.conv2(x)
            return x



In [27]:
class MRI_resVAE(nn.Module):
    
    def __init__(self,feature_size,latent_size, n_channels=1, gf_dim = 16):
        
        super(MRI_resVAE, self).__init__()
        
        self.feature_size = feature_size
        self.latent_size = latent_size
        self.n_channels = n_channels
        self.gf_dim = gf_dim

        # encoder
        #self.pool= 0
        
        self.convd1_block = nn.Sequential(
            nn.Conv2d(n_channels, gf_dim, 3, stride=1, padding=1),
            nn.BatchNorm2d(gf_dim),
            nn.LeakyReLU(0.2, inplace=True))

        self.resd1 = ResBlockDown(gf_dim * 1, gf_dim * 1)
        self.resd2 = ResBlockDown(gf_dim * 1, gf_dim * 2)
        self.resd3 = ResBlockDown(gf_dim * 2, gf_dim * 4)
        self.resd4 = ResBlockDown(gf_dim * 4, gf_dim * 8)
        self.resd5 = ResBlockDown(gf_dim * 8, gf_dim * 16)

        self.resd6 = ResBlockDown(gf_dim * 16, gf_dim * 32)
        self.resd7 = ResBlockDown(gf_dim * 32, gf_dim * 64)#, act=False)
        #self.resd2_stdev = ResBlockDown(gf_dim * 32, gf_dim * 64, act=False)
        
        self.flatten = nn.Flatten()
        self.dense_mean = nn.Linear(self.feature_size, self.latent_size)  
        self.dense_log_var = nn.Linear(self.feature_size, self.latent_size)
        
        #decoder
        self.dense = nn.Linear(self.latent_size, self.feature_size)

        self.resu0 = ResBlockUp(gf_dim*64, gf_dim * 32)
        self.resu1 = ResBlockUp(gf_dim*32, gf_dim * 16)
        self.resu2 = ResBlockUp(gf_dim*16, gf_dim * 8)
        self.resu3 = ResBlockUp(gf_dim*8, gf_dim * 4)
        self.resu4 = ResBlockUp(gf_dim*4, gf_dim * 2)
        self.resu5 = ResBlockUp(gf_dim*2, gf_dim * 1)
        self.resu6 = ResBlockUp(gf_dim*1, gf_dim * 1)
        self.convu_1_block = nn.Sequential(
            nn.Conv2d(gf_dim, gf_dim, 3, stride=1, padding=1),
            #nn.BatchNorm2d(gf_dim),
            nn.LeakyReLU()) #0.2, inplace=True))


        self.convu2 = nn.Sequential(
            nn.Conv2d(gf_dim, n_channels, 3, stride=1, padding=1),
            nn.ReLU())




    def encode(self, x): # with condition - def encode(self, x, c):
        '''
        x: input scan (bs, 1, 197, 233, 189)
        c: condition (bs, class_size)
        '''
        #print(x.shape)
        image_padder = ImagePadding_4dtensor(x)
        
        # Target size [256, 256, 256]
        #target_size = (243, 243, 243)
        target_size = (256, 256)

        # Get the padded image
        padded_image = image_padder.pad_to_size(target_size)
        #print(padded_image.shape)
        #print('encoding')
        
        conv1 = self.convd1_block(padded_image)
        z = self.resd1(conv1)
        z = self.resd2(z)
        z = self.resd3(z)
        z = self.resd4(z)
        z = self.resd5(z)
        z = self.resd6(z)
        z = self.resd7(z)
        #print(z.shape)
        self.remember_shape = z.shape
        z = self.flatten(z)
        #print(x.shape)
        z_mu = self.dense_mean(z)
        z_var = self.dense_log_var(z)
        z = self.reparameterize(z_mu, z_var)

       
        
        return z_mu, z_var

    def reparameterize(self, mu, logvar):
        #print('reparameterize')
        std = torch.exp(0.5*logvar)
        eps = torch.randn_like(std)
        return mu + eps*std

    def decode(self, z): # def decode(self, z, c):
        '''
        z: (bs, latent_size)
        c: (bs, class_size)
        '''
        #print('decoder')
        #inputs = torch.cat([z, c], 1) # (bs, latent_size+class_size)
        x = self.dense(z)
        #print(x.shape)
        x = torch.reshape(x,(self.remember_shape[0],self.remember_shape[1],self.remember_shape[2],self.remember_shape[3]))
        
        x = self.resu0(x)
        x = self.resu1(x)
        x = self.resu2(x)
        x = self.resu3(x)
        x = self.resu4(x)
        x = self.resu5(x)
        x = self.resu6(x)
        x = self.convu_1_block(x)
        x = self.convu2(x)
        
        image_cropper = ImageCropping_4dtensor(x)
        
        # Target size [182, 218, 182]
        target_size = (182, 218)

        # Get the cropped image
        cropped_image = image_cropper.crop_to_size(target_size)
        #print(cropped_image.shape)
        return cropped_image #torch.cat([cropped_image_pet, cropped_image_mri], dim=1)

    def forward(self, x): #def forward(self, x, c):
        
        mu, logvar = self.encode(x) #self.encode(x, c)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar #self.decode(z, c), mu, logvar


In [28]:
'''device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = MRI_resVAE(1024*2*2, latent_size =1).to(device)
sum([param.nelement() for param in model.parameters()])'''

'device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")\nmodel = MRI_resVAE(1024*2*2, latent_size =1).to(device)\nsum([param.nelement() for param in model.parameters()])'

In [29]:
'''from torchsummary import summary
help(summary)
summary(model, [(1,182, 218)] , device='cpu') #[(2,182, 218, 182),(2,)]'''

"from torchsummary import summary\nhelp(summary)\nsummary(model, [(1,182, 218)] , device='cpu') #[(2,182, 218, 182),(2,)]"

In [30]:
subprocess.run(['jupyter', 'nbconvert', '--to', 'script', 'addl_models_bimodel_pytorch.ipynb'], check=True)

[NbConvertApp] Converting notebook addl_models_bimodel_pytorch.ipynb to script
[NbConvertApp] Writing 51482 bytes to addl_models_bimodel_pytorch.py


CompletedProcess(args=['jupyter', 'nbconvert', '--to', 'script', 'addl_models_bimodel_pytorch.ipynb'], returncode=0)